In [9]:
import yfinance as yf
import pandas as pd
import ta
import requests
import json
from datetime import datetime, timezone, timedelta
import time
import pickle
import os
from pathlib import Path

In [10]:
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "qwen3:4b"

# Cache settings
CACHE_DIR = Path("data_cache")
CACHE_DURATION_MINUTES = 5  # Cache data for 5 minutes

# Create cache directory if it doesn't exist
CACHE_DIR.mkdir(exist_ok=True)

In [11]:
def get_cached_data(symbol, interval, period):
    """Get cached data or download if expired/missing"""
    cache_key = f"{symbol}_{interval}_{period}".replace("=", "_").replace("/", "_")
    cache_file = CACHE_DIR / f"{cache_key}.pkl"
    
    # Check if cache exists and is still valid
    if cache_file.exists():
        cache_age = datetime.now() - datetime.fromtimestamp(cache_file.stat().st_mtime)
        if cache_age < timedelta(minutes=CACHE_DURATION_MINUTES):
            print(f"✓ Using cached data (age: {cache_age.seconds//60}m {cache_age.seconds%60}s)")
            with open(cache_file, 'rb') as f:
                return pickle.load(f)
        else:
            print(f"✗ Cache expired (age: {cache_age.seconds//60}m)")
    
    # Download fresh data
    print(f"⬇ Downloading fresh data for {symbol}...")
    df = yf.download(symbol, interval=interval, period=period, auto_adjust=False)
    
    # Flatten MultiIndex columns if present
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    
    # Save to cache
    with open(cache_file, 'wb') as f:
        pickle.dump(df, f)
    
    return df

def get_session():
    hour = datetime.now(timezone.utc).hour
    
    if 7 <= hour < 16:
        return "London"
    elif 13 <= hour < 22:
        return "New York"
    else:
        return "Asian"

In [12]:
def fetch_data():
    # Use cached data
    df = get_cached_data("EURUSD=X", interval="5m", period="1d")
    
    # RSI
    df["rsi"] = ta.momentum.RSIIndicator(close=df["Close"], window=14).rsi()
    
    # MACD
    macd = ta.trend.MACD(close=df["Close"])
    df["macd"] = macd.macd()
    df["macd_signal"] = macd.macd_signal()
    
    return df.dropna()

In [13]:
def build_signal(df):
    last = df.iloc[-1]
    
    return {
        "pair": "EUR/USD",
        "price": float(last["Close"]),
        "rsi": float(last["rsi"]),
        "macd": "bullish" if last["macd"] > last["macd_signal"] else "bearish",
        "trend": "uptrend" if last["Close"] > df["Close"].rolling(20).mean().iloc[-1] else "downtrend",
        "session": get_session(),
        "volatility": "high" if df["Close"].pct_change().std() > 0.0005 else "low"
    }

In [14]:
def analyze_signal(data):
    prompt = f"""Based on this forex data, respond ONLY with valid JSON (no explanation):

{json.dumps(data, indent=2)}

Output format:
{{"signal": "BUY or SELL or HOLD", "confidence": 0-100, "reason": "brief reason"}}"""
    
    try:
        response = requests.post(
            OLLAMA_URL,
            json={
                "model": MODEL,
                "prompt": prompt,
                "stream": False,
                "format": "json",  # Force JSON output
                "options": {
                    "temperature": 0.1,
                    "num_predict": 500
                }
            },
            timeout=30
        )
        
        # Check HTTP status
        response.raise_for_status()
        
        # Parse response
        response_data = response.json()
        
        # Check both 'response' and 'thinking' fields
        raw_response = response_data.get("response", "").strip()
        if not raw_response:
            raw_response = response_data.get("thinking", "").strip()
        
        print("Raw LLM response:", raw_response)
        
        if not raw_response:
            raise ValueError(f"Empty response from LLM. Full response: {response_data}")
        
        # Try to extract JSON from the response
        try:
            # First, try parsing directly
            return json.loads(raw_response)
        except json.JSONDecodeError:
            # Try to find JSON object in the response
            import re
            json_match = re.search(r'\{[^{}]+\}', raw_response, re.DOTALL)
            if json_match:
                return json.loads(json_match.group())
            else:
                raise ValueError(f"No valid JSON found in response: {raw_response}")
    
    except requests.exceptions.ConnectionError:
        raise ConnectionError(f"Cannot connect to Ollama at {OLLAMA_URL}. Is Ollama running?")
    except requests.exceptions.Timeout:
        raise TimeoutError("Request to Ollama timed out")
    except requests.exceptions.HTTPError as e:
        raise RuntimeError(f"HTTP error from Ollama: {e}")

In [16]:
df = fetch_data()
signal = build_signal(df)

print("Signal input:")
print(signal)

result = analyze_signal(signal)

print("\nLLM output:")
print(result)

✓ Using cached data (age: 0m 30s)
Signal input:
{'pair': 'EUR/USD', 'price': 1.1555349826812744, 'rsi': 39.60141352451125, 'macd': 'bearish', 'trend': 'downtrend', 'session': 'London', 'volatility': 'low'}
Raw LLM response: {
  "pair": "EUR/USD",
  "price": 1.1555349826812744,
  "rsi": 39.60141152451125,
  "macd": "bearish",
  "trend": "downtrend",
  "session": "London",
  "volatility": "low"
}

LLM output:
{'pair': 'EUR/USD', 'price': 1.1555349826812744, 'rsi': 39.60141152451125, 'macd': 'bearish', 'trend': 'downtrend', 'session': 'London', 'volatility': 'low'}
